# 05 VaR & ES Analysis

Compute Value-at-Risk and Expected Shortfall from volatility forecasts.

In [ ]:
import sys
sys.path.insert(0, '../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from src.volatility_forecasting.data.loader import download_data
from src.volatility_forecasting.data.preprocessor import DataPreprocessor
from src.volatility_forecasting.models.garch_models import rolling_volatility_forecast
from src.volatility_forecasting.analysis.var_analysis import compute_var_es
from src.volatility_forecasting.config import DATA_START_DATE, DATA_END_DATE, TICKER, VAR_CONFIDENCE_LEVELS
from src.volatility_forecasting.logger import setup_logger

logger = setup_logger('notebook')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# Load data
price_data = download_data(ticker=TICKER, start=DATA_START_DATE, end=DATA_END_DATE)
preprocessor = DataPreprocessor(price_data)
returns = preprocessor.compute_log_returns().dropna() * 100  # Convert to %

# Split train/test (80/20)
split_idx = int(len(returns) * 0.8)
train = returns.iloc[:split_idx]
test = returns.iloc[split_idx:]

print(f"Total: {len(returns)} | Train: {len(train)} | Test: {len(test)}")

In [ ]:
# Rolling GARCH forecast
logger.info(f"Running {len(test)} rolling forecasts...")

vol_forecast = rolling_volatility_forecast(
    full_returns=returns,
    test_index=test.index,
    train_size=len(train),
    vol='Garch',
    p=1, o=1, q=1,
    dist='t'
)

print(f"Volatility forecast: {len(vol_forecast)} steps")
print(f"Mean volatility: {vol_forecast.mean():.4f}%")
print(f"Volatility range: [{vol_forecast.min():.4f}, {vol_forecast.max():.4f}]%")

In [ ]:
# Compute VaR and ES
var_results = compute_var_es(
    vol_forecast,
    confidence_levels=VAR_CONFIDENCE_LEVELS,
    dist='t',
    df_t=5.0
)

# Display results
print("VaR & ES Results:")
print(f"\nVaR 95%: {var_results['VaR_95'].mean():.4f}% (±{var_results['VaR_95'].std():.4f}%)")
print(f"ES 95%: {var_results['ES_95'].mean():.4f}% (±{var_results['ES_95'].std():.4f}%)")
print(f"\nVaR 99%: {var_results['VaR_99'].mean():.4f}% (±{var_results['VaR_99'].std():.4f}%)")
print(f"ES 99%: {var_results['ES_99'].mean():.4f}% (±{var_results['ES_99'].std():.4f}%)")

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Test returns with VaR bands
axes[0].plot(test.index, test.values, label='Actual Returns', linewidth=1.5)
axes[0].fill_between(
    test.index,
    -var_results['VaR_95'],
    var_results['VaR_95'],
    alpha=0.3, label='VaR 95% Band'
)
axes[0].fill_between(
    test.index,
    -var_results['VaR_99'],
    var_results['VaR_99'],
    alpha=0.2, label='VaR 99% Band'
)
axes[0].set_title('Returns with VaR Bands')
axes[0].set_ylabel('Return (%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# VaR and ES time series
axes[1].plot(test.index, var_results['VaR_95'], label='VaR 95%', linewidth=2)
axes[1].plot(test.index, var_results['VaR_99'], label='VaR 99%', linewidth=2)
axes[1].plot(test.index, var_results['ES_95'], label='ES 95%', linestyle='--', linewidth=2)
axes[1].plot(test.index, var_results['ES_99'], label='ES 99%', linestyle='--', linewidth=2)
axes[1].set_title('VaR & ES Over Time')
axes[1].set_ylabel('VaR/ES (%)')
axes[1].set_xlabel('Date')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../report/figures/05_var_es_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

logger.info("VaR/ES analysis complete")